# Train Data Analysis — Part 2

## Notebook Overview

In this notebook, we continued the exploratory analysis of the **training dataset** by examining **clinical measurements, physiological indicators, temporal variables, and emergency department outcomes**.

The analysis included:

- Summary statistics of **vital signs** (blood pressure, heart rate, respiratory rate, temperature, oxygen saturation).
- Examination of **clinical scores** such as **GCS, pain score, and NEWS2**.
- Analysis of **body measurements** including **weight, height, and BMI**.
- Review of **derived clinical indicators** like **shock index**.
- Exploration of **time-related features** describing when patients arrive at the emergency department.
- Summary of **ED outcomes**, including **disposition** and **length of stay**.

This analysis helps understand **patient physiological status, temporal patterns of emergency visits, and outcome distributions**, providing important context before moving to deeper modeling and feature engineering.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as pyplot
import seaborn as sns


In [2]:
df_train = pd.read_csv("../data/train.csv")
df_train.head()

,patient_id,site_id,triage_nurse_id,arrival_mode,arrival_hour,arrival_day,arrival_month,arrival_season,shift,age,...,gcs_total,pain_score,weight_kg,height_cm,bmi,shock_index,news2_score,disposition,ed_los_hours,triage_acuity
0,TG-UXRGA9UCO,SITE-TMP-01,NURSE-0033,walk-in,6,Monday,5,spring,morning,43,...,14,7,52.3,165.4,19.1,0.725,8,discharged,7.35,2
1,TG-B19DBBS2G,SITE-HEL-01,NURSE-0001,walk-in,6,Thursday,4,spring,morning,72,...,15,-1,73.3,164.4,27.1,0.739,1,discharged,0.70,5
2,TG-GZ97W7M6V,SITE-HEL-02,NURSE-0005,walk-in,8,Saturday,4,spring,morning,82,...,15,3,77.1,183.7,22.8,0.798,2,discharged,0.63,5
3,TG-THIB2TN9Q,SITE-HEL-02,NURSE-0026,police,7,Sunday,3,spring,morning,50,...,15,7,49.6,172.6,16.6,0.812,2,discharged,1.99,3
4,TG-J3U3LQ2QY,SITE-HEL-02,NURSE-0044,walk-in,5,Tuesday,5,spring,night,62,...,15,4,71.9,173.4,23.9,0.812,2,transferred,3.58,3


In [3]:
df_info = pd.DataFrame({
    "Column": df_train.columns,
    "Non-Null Count": df_train.count().values,
    "Null Count": df_train.isnull().sum().values,
    "Data Type": df_train.dtypes.values
})

df_info

,Column,Non-Null Count,Null Count,Data Type
0,patient_id,80000,0,object
1,site_id,80000,0,object
2,triage_nurse_id,80000,0,object
3,arrival_mode,80000,0,object
4,arrival_hour,80000,0,int64
5,arrival_day,80000,0,object
6,arrival_month,80000,0,int64
7,arrival_season,80000,0,object
8,shift,80000,0,object
9,age,80000,0,int64


In [4]:
# Identifier columns
id_cols = [
    "patient_id",
    "site_id",
    "triage_nurse_id"
]

# Time-related features
time_cols = [
    "arrival_hour",
    "arrival_day",
    "arrival_month",
    "arrival_season",
    "shift"
]

# Demographic categorical features
demographic_cat = [
    "sex",
    "language",
    "insurance_type",
    "age_group"
]

# Visit context categorical features
context_cat = [
    "arrival_mode",
    "transport_origin",
    "pain_location",
    "mental_status_triage",
    "chief_complaint_system"
]

# Count / utilization features
count_cols = [
    "num_prior_ed_visits_12m",
    "num_prior_admissions_12m",
    "num_active_medications",
    "num_comorbidities"
]

# Continuous vital signs
vital_continuous = [
    "systolic_bp",
    "diastolic_bp",
    "mean_arterial_pressure",
    "pulse_pressure",
    "heart_rate",
    "respiratory_rate",
    "temperature_c",
    "spo2"
]

# Clinical scores (ordinal / discrete)
clinical_scores = [
    "gcs_total",
    "pain_score",
    "news2_score"
]

# Anthropometric continuous features
body_measurements = [
    "weight_kg",
    "height_cm",
    "bmi"
]

# Derived physiological metrics
derived_features = [
    "shock_index"
]

# Outcome / target variables
target_cols = [
    "triage_acuity"
]

# Post-visit outcomes (should not be used as predictors if predicting triage)
post_visit_outcomes = [
    "disposition",
    "ed_los_hours"
]

# `Continuous Features`

In [5]:
for i in vital_continuous:
    print(f"Summary statistics for {i}:")
    print(df_train[i].describe())
    print("\n") 

Summary statistics for systolic_bp:
count    75854.000000
mean       121.612599
std         24.221898
min         40.000000
25%        106.200000
50%        123.100000
75%        138.300000
max        226.900000
Name: systolic_bp, dtype: float64


Summary statistics for diastolic_bp:
count    75854.000000
mean        74.452339
std         14.285613
min         20.000000
25%         65.500000
50%         75.300000
75%         84.200000
max        134.800000
Name: diastolic_bp, dtype: float64


Summary statistics for mean_arterial_pressure:
count    75854.000000
mean        90.172434
std         14.174033
min         30.700000
25%         81.900000
50%         91.900000
75%        100.000000
max        145.100000
Name: mean_arterial_pressure, dtype: float64


Summary statistics for pulse_pressure:
count    75854.000000
mean        47.160261
std         24.253618
min        -51.000000
25%         31.100000
50%         47.200000
75%         63.400000
max        163.700000
Name: pulse_press

## Vital Signs Summary Statistics

This section analyzes the **distribution of key physiological vital signs** recorded at triage.  
Vital signs are critical indicators of **patient stability and immediate clinical risk**, and they play an important role in determining emergency triage severity.

---

## Systolic Blood Pressure (SBP)

| Statistic | Value |
|---|---|
| Mean | 121.61 |
| Median | 123.10 |
| Std | 24.22 |
| Min | 40.00 |
| Max | 226.90 |

**Observations**

- The average SBP (~121 mmHg) falls within the **normal adult range**.
- Very low values (40 mmHg) suggest possible **severe hypotension or measurement anomalies**.
- Extremely high values (>200 mmHg) may indicate **hypertensive emergencies**.

---

## Diastolic Blood Pressure (DBP)

| Statistic | Value |
|---|---|
| Mean | 74.45 |
| Median | 75.30 |
| Std | 14.29 |
| Min | 20.00 |
| Max | 134.80 |

**Observations**

- Mean DBP (~74 mmHg) is within the **normal physiological range**.
- Very low values may reflect **shock or circulatory instability**.
- High values could indicate **severe hypertension**.

---

## Mean Arterial Pressure (MAP)

| Statistic | Value |
|---|---|
| Mean | 90.17 |
| Median | 91.90 |
| Std | 14.17 |
| Min | 30.70 |
| Max | 145.10 |

**Observations**

- MAP around **90 mmHg** suggests generally adequate organ perfusion.
- MAP below **65 mmHg** may indicate **risk of organ hypoperfusion**.

---

## Pulse Pressure

| Statistic | Value |
|---|---|
| Mean | 47.16 |
| Median | 47.20 |
| Std | 24.25 |
| Min | -51.00 |
| Max | 163.70 |

**Observations**

- Typical pulse pressure is around **40–60 mmHg**.
- Negative values likely reflect **measurement inconsistencies or data noise**.
- Very high values may indicate **arterial stiffness or cardiovascular abnormalities**.

---

## Heart Rate

| Statistic | Value |
|---|---|
| Mean | 91.86 |
| Median | 89.60 |
| Std | 19.49 |
| Min | 30.00 |
| Max | 207.70 |

**Observations**

- Mean heart rate (~92 bpm) is slightly elevated compared to typical resting values.
- Extremely low heart rates may represent **bradycardia**.
- Very high values (>200 bpm) may indicate **tachyarrhythmia or severe physiological stress**.

---

## Respiratory Rate

| Statistic | Value |
|---|---|
| Mean | 18.33 |
| Median | 17.30 |
| Std | 4.65 |
| Min | 8.00 |
| Max | 51.50 |

**Observations**

- The average respiratory rate (~18 breaths/min) is within the **normal adult range**.
- High respiratory rates may indicate **respiratory distress or infection**.

---

## Body Temperature

| Statistic | Value |
|---|---|
| Mean | 37.63°C |
| Median | 37.50°C |
| Std | 0.86 |
| Min | 35.10°C |
| Max | 41.80°C |

**Observations**

- Average temperature (~37.6°C) is close to **normal body temperature**.
- High temperatures (>39°C) may indicate **severe infection or systemic inflammation**.

---

## Oxygen Saturation (SpO₂)

| Statistic | Value |
|---|---|
| Mean | 95.79% |
| Median | 97.00% |
| Std | 4.31 |
| Min | 60.40% |
| Max | 100.00% |

**Observations**

- Most patients maintain **adequate oxygen saturation levels**.
- Very low values (<90%) may indicate **respiratory compromise or hypoxia**.

---

## Summary

Overall, the vital signs appear **clinically realistic**, with most values falling within expected physiological ranges.  
However, the presence of **extreme values and missing measurements** suggests potential data variability that may require **cleaning or outlier handling during preprocessing**.



`  min pulse_pressure = -51    A small number of physiologically implausible values are observed, particularly negative pulse pressure values, which likely result from measurement or calculation inconsistencies. Additionally, several extreme vital sign values appear that may represent either severe clinical conditions or measurement noise.`

# `Clinical Scores Features`

In [8]:
for i in clinical_scores:
    print(f"Summary statistics for {i}:")
    print(df_train[i].describe())
    print("\n")

Summary statistics for gcs_total:
count    80000.000000
mean        14.152125
std          2.145761
min          3.000000
25%         15.000000
50%         15.000000
75%         15.000000
max         15.000000
Name: gcs_total, dtype: float64


Summary statistics for pain_score:
count    80000.000000
mean         4.505975
std          3.359763
min         -1.000000
25%          2.000000
50%          5.000000
75%          7.000000
max         10.000000
Name: pain_score, dtype: float64


Summary statistics for news2_score:
count    80000.000000
mean         3.426825
std          4.258565
min          0.000000
25%          0.000000
50%          2.000000
75%          5.000000
max         17.000000
Name: news2_score, dtype: float64




## Clinical Assessment Scores

This section summarizes key **clinical scoring systems used during triage**.  
These scores help clinicians rapidly assess **neurological status, pain severity, and physiological deterioration risk**.

---

## Glasgow Coma Scale (GCS)

| Statistic | Value |
|---|---|
| Mean | 14.15 |
| Median | 15 |
| Std | 2.15 |
| Min | 3 |
| Max | 15 |

**Observations**

- Most patients have **GCS = 15**, indicating **normal consciousness**.
- Lower values indicate **impaired neurological status**.
- The minimum score of **3** corresponds to **deep unconsciousness or coma**.

---

## Pain Score

| Statistic | Value |
|---|---|
| Mean | 4.51 |
| Median | 5 |
| Std | 3.36 |
| Min | -1 |
| Max | 10 |

**Observations**

- Pain scores range from **0–10**, representing increasing pain intensity.
- The average score (~4.5) suggests **moderate pain levels** in many patients.
- The value **-1 likely indicates missing or unrecorded pain scores**.

---

## NEWS2 Score

| Statistic | Value |
|---|---|
| Mean | 3.43 |
| Median | 2 |
| Std | 4.26 |
| Min | 0 |
| Max | 17 |

**Observations**

- NEWS2 measures **physiological deterioration risk** based on vital signs.
- Most patients have **low scores**, indicating **low immediate risk**.
- Higher scores (≥7) may indicate **serious clinical deterioration requiring urgent care**.

---

## Summary

These clinical scores capture **critical patient status during triage**:

- **GCS** reflects neurological consciousness.
- **Pain score** indicates patient discomfort and symptom severity.
- **NEWS2 score** summarizes physiological instability.

Together, these features provide strong signals for **predicting triage acuity and clinical severity** in emergency department settings.

`Note pain_score = -1 → missing indicator gcs_total highly skewed (most = 15) very high NEWS2 values may indicate severe cases`

# `Body Measurements Features`


In [9]:
for i in body_measurements:
    print(f"Summary statistics for {i}:")
    print(df_train[i].describe())
    print("\n")

Summary statistics for weight_kg:
count    80000.000000
mean        74.453355
std         21.339829
min          2.000000
25%         62.000000
50%         76.000000
75%         88.800000
max        148.500000
Name: weight_kg, dtype: float64


Summary statistics for height_cm:
count    80000.000000
mean       168.623680
std         16.592256
min         45.000000
25%        163.400000
50%        171.100000
75%        178.100000
max        210.000000
Name: height_cm, dtype: float64


Summary statistics for bmi:
count    80000.000000
mean        26.359772
std          7.666452
min         10.000000
25%         21.300000
50%         26.000000
75%         30.900000
max         65.000000
Name: bmi, dtype: float64




## Anthropometric Measurements

This section summarizes **body measurement features** including weight, height, and Body Mass Index (BMI).  
These variables provide insight into **patient body composition and nutritional status**, which may influence disease risk and clinical outcomes in emergency care.

---

## Weight (kg)

| Statistic | Value |
|---|---|
| Mean | 74.45 kg |
| Median | 76.00 kg |
| Std | 21.34 |
| Min | 2.00 kg |
| Max | 148.50 kg |

**Observations**

- The average weight (~74 kg) is within a typical adult range.
- The minimum value of **2 kg likely represents pediatric patients or potential data entry anomalies**.
- Very high weights (>120 kg) indicate **severe obesity in a subset of patients**.

---

## Height (cm)

| Statistic | Value |
|---|---|
| Mean | 168.62 cm |
| Median | 171.10 cm |
| Std | 16.59 |
| Min | 45.00 cm |
| Max | 210.00 cm |

**Observations**

- Average height (~169 cm) aligns with typical adult populations.
- Extremely low values such as **45 cm likely correspond to pediatric patients**.
- Maximum values (~210 cm) represent very tall individuals but remain within possible biological limits.

---

## Body Mass Index (BMI)

| Statistic | Value |
|---|---|
| Mean | 26.36 |
| Median | 26.00 |
| Std | 7.67 |
| Min | 10.00 |
| Max | 65.00 |

**Observations**

- The average BMI (~26) falls within the **overweight category**.
- BMI values below **18.5 indicate underweight patients**, which may be associated with malnutrition or chronic illness.
- Very high BMI values (>40) suggest **severe obesity**, which increases risk for cardiovascular and metabolic complications.

---

## Summary

Anthropometric features reveal **wide variation in patient body composition**, reflecting the presence of both **pediatric and adult populations** as well as individuals across different weight categories.

These variables may influence **disease risk, medication dosing, and physiological stress responses**, making them useful contextual features for **triage severity prediction**.


`### ⚠️ Potential Red Flags in Anthropometric Data`

| Feature | Value | Note |
|---|---|---|
| **weight_kg (min)** | 2 kg | Extremely low; likely newborn/pediatric entry or possible data entry error. |
| **height_cm (min)** | 45 cm | Very short height, consistent with infants but unusual for general ED population. |
| **bmi (min)** | 10 | Indicates severe underweight; could represent malnutrition, pediatric case, or measurement issue. |
| **bmi (max)** | 65 | Extremely high BMI indicating severe morbid obesity. |

**Conclusion:**  
The dataset likely contains **both pediatric and adult patients**, which explains extreme values in weight, height, and BMI. However, these outliers should be **validated or handled carefully during preprocessing** to avoid distortion in model training.

# `Derived Features`


In [11]:
df_train[derived_features].describe()

,shock_index
count,75854.000000
mean,0.810019
std,0.327007
min,0.190000
25%,0.601000
50%,0.724000
75%,0.920000
max,4.768000


## Shock Index (SI)

| Statistic | Value |
|---|---|
| Count | 75,854 |
| Mean | 0.81 |
| Std | 0.33 |
| Min | 0.19 |
| 25% | 0.60 |
| Median | 0.72 |
| 75% | 0.92 |
| Max | 4.77 |

### Clinical Context


It is used to assess **hemodynamic instability and risk of shock**.

Typical interpretation:

| SI Range | Interpretation |
|---|---|
| 0.5 – 0.7 | Normal |
| 0.7 – 0.9 | Mild physiological stress |
| > 1.0 | Possible shock / circulatory compromise |
| > 1.3 | High risk of severe shock |

### ⚠️ Potential Red Flags

| Observation | Value | Note |
|---|---|---|
| Very high SI | 4.77 | Extremely abnormal; may indicate severe shock, extreme vitals, or calculation error. |
| High upper quartile | 0.92 | Indicates a notable portion of patients approaching hemodynamic instability. |

**Conclusion**

Most patients fall within **normal to mildly elevated shock index ranges**, but extreme values suggest **possible severe circulatory compromise or measurement anomalies**, which should be carefully inspected during preprocessing.

# Post Visit Outcomes Features

In [12]:
for i in post_visit_outcomes:
    print(f"Summary statistics for {i}:")
    print(df_train[i].describe())
    print("\n")

Summary statistics for disposition:
count          80000
unique             7
top       discharged
freq           39028
Name: disposition, dtype: object


Summary statistics for ed_los_hours:
count    80000.000000
mean         3.498276
std          2.443958
min          0.000000
25%          1.580000
50%          3.000000
75%          4.860000
max         17.510000
Name: ed_los_hours, dtype: float64




In [13]:
for i in post_visit_outcomes:
    print(f"Summary statistics for {i}:")
    print(df_train[i].value_counts())
    print("\n")

Summary statistics for disposition:
disposition
discharged     39028
admitted       24601
transferred     5203
observation     4337
lwbs            3656
lama            2764
deceased         411
Name: count, dtype: int64


Summary statistics for ed_los_hours:
ed_los_hours
1.22     193
1.24     191
1.31     191
1.01     191
1.55     186
        ... 
13.73      1
12.08      1
12.51      1
13.60      1
14.26      1
Name: count, Length: 1404, dtype: int64




## Emergency Department Outcomes

This section summarizes **post-visit outcomes** including patient disposition and emergency department length of stay (ED LOS).

---

## Disposition

| Outcome | Count |
|---|---|
| discharged | 39,028 |
| admitted | 24,601 |
| transferred | 5,203 |
| observation | 4,337 |
| lwbs (left without being seen) | 3,656 |
| lama (left against medical advice) | 2,764 |
| deceased | 411 |

### Observations

- Nearly **49% of patients were discharged**, indicating many visits were non-critical.
- About **31% required hospital admission**, suggesting a substantial number of moderate to severe cases.
- A small number of patients **left before treatment or against medical advice**.
- **Deaths are rare (~0.5%)**, which is expected in general emergency department data.

---

## Emergency Department Length of Stay (ED LOS)

| Statistic | Value |
|---|---|
| Mean | 3.50 hours |
| Median | 3.00 hours |
| Std | 2.44 |
| Min | 0.00 |
| Max | 17.51 |

### Observations

- The median stay of **~3 hours** suggests typical emergency department throughput.
- The maximum stay of **17.5 hours** indicates some patients required extended observation or complex care.
- A LOS of **0 hours may represent immediate transfer, rapid discharge, or data rounding**.

---

### ⚠️ Modeling Note

Both **`disposition`** and **`ed_los_hours`** occur **after triage**, meaning they are **post-outcome variables**.

Using them as predictors for **triage_acuity** would introduce **data leakage**, since they contain information about events that happen after the triage decision.

Therefore, these variables should **not be used as input features during model training**.

# `Time Features`

In [14]:
for col in time_cols:
    print(f"\nValue counts for {col}:")
    print(df_train[col].value_counts())


Value counts for arrival_hour:
arrival_hour
16    3436
1     3434
2     3405
10    3405
8     3392
0     3387
22    3372
9     3369
5     3367
7     3350
3     3347
6     3338
14    3326
19    3321
17    3316
15    3313
21    3299
11    3299
4     3299
13    3268
12    3260
18    3250
20    3232
23    3215
Name: count, dtype: int64

Value counts for arrival_day:
arrival_day
Monday       11539
Thursday     11534
Saturday     11469
Wednesday    11437
Friday       11407
Sunday       11358
Tuesday      11256
Name: count, dtype: int64

Value counts for arrival_month:
arrival_month
1     6752
12    6730
9     6710
11    6704
3     6699
5     6694
8     6665
7     6664
10    6640
4     6620
6     6572
2     6550
Name: count, dtype: int64

Value counts for arrival_season:
arrival_season
autumn    20054
winter    20032
spring    20013
summer    19901
Name: count, dtype: int64

Value counts for shift:
shift
morning      26681
night        20239
afternoon    19962
evening      13118
Name: count,

## Temporal Arrival Patterns

This section analyzes **time-related variables** describing when patients arrive at the emergency department.  
These features help identify **daily workload patterns, weekly trends, and seasonal variations in patient arrivals**.

---

## Arrival Hour

| Hour | Count |
|---|---|
| Range | 0–23 |
| Highest | 16 (3,436 visits) |
| Lowest | 23 (3,215 visits) |

**Observations**

- Patient arrivals are **fairly evenly distributed across the 24-hour period**.
- Slight peaks appear during **late afternoon and early morning hours**.
- The relatively uniform distribution suggests **consistent ED demand throughout the day**, which is common in large hospitals.

---

## Arrival Day

| Day | Count |
|---|---|
| Monday | 11,539 |
| Thursday | 11,534 |
| Saturday | 11,469 |
| Wednesday | 11,437 |
| Friday | 11,407 |
| Sunday | 11,358 |
| Tuesday | 11,256 |

**Observations**

- Patient arrivals are **almost evenly distributed across the week**.
- Monday shows a slightly higher volume, which may reflect **patients delaying care over the weekend**.
- No extreme weekday imbalance is observed.

---

## Arrival Month

| Month | Approx Count |
|---|---|
| Range | ~6,550 – 6,750 |

**Observations**

- Monthly arrivals are **very evenly distributed throughout the year**.
- There is no strong monthly spike, suggesting **stable yearly emergency department demand**.

---

## Arrival Season

| Season | Count |
|---|---|
| Autumn | 20,054 |
| Winter | 20,032 |
| Spring | 20,013 |
| Summer | 19,901 |

**Observations**

- The dataset is **almost perfectly balanced across seasons**.
- This suggests the dataset was likely **constructed to avoid seasonal bias**.

---

## Shift Distribution

| Shift | Count |
|---|---|
| Morning | 26,681 |
| Night | 20,239 |
| Afternoon | 19,962 |
| Evening | 13,118 |

**Observations**

- **Morning shift has the highest patient volume**, indicating peak operational workload.
- Evening shift has the **lowest number of arrivals**.
- These patterns may reflect **hospital scheduling and patient behavior patterns**.

---

## Summary

Temporal variables show **balanced distributions across hours, days, months, and seasons**, suggesting the dataset captures **consistent emergency department activity without strong temporal bias**.  
These features may still provide useful signals when combined with **clinical variables and triage severity outcomes**.


`Temporal features appear **highly balanced across days, months, and seasons**, which may indicate that the dataset was **designed to reduce temporal bias**. While this improves modeling fairness, it may also **reduce real-world temporal variability signals**`